In [ ]:
# Cell 1: Imports and parameters
import os
os.environ['JAX_PLATFORMS'] = 'cpu'  # Force JAX to use CPU

import jax
import jax.numpy as jnp
from jax.experimental.ode import odeint
import numpy as np

# Verify we're using CPU
print(f"JAX backend: {jax.default_backend()}")
print(f"Available devices: {jax.devices()}")

jax.config.update("jax_enable_x64", True)


In [ ]:
# Setup output directory for saving plots
import os
from datetime import datetime

# Get today's date in YYYY-MM-DD format
today = datetime.now().strftime("%Y-%m-%d")
output_dir = f"out/{today}/method_comparison_damping/"

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory created: {output_dir}")

save_flag = True

# Define Model

In [ ]:

# System size and mass
N = 5   # number of masses
mass = 1.0
gamma = 0.0 # damping coefficient

atol = 1e-12
rtol = 1e-12

# Unpack parameters: p[0:N+1] = k_i (spring stiffnesses), p[N+1:] = beta (Duffing coeffs)
# For N masses, we have N+1 springs: wall-mass0, mass0-mass1, ..., mass(N-2)-mass(N-1), mass(N-1)-wall
def unpack_params(p):
    k_springs = p[:N+1]  # N+1 spring constants
    beta = p[N+1:]       # N Duffing coefficients
    return k_springs, beta

In [ ]:
# Cell 2: Forcing and system dynamics
def forcing(t):
    # sinusoidal drive on mass index 0
    return 1.*jnp.array([jnp.sin(2.0 * t)] + [0.0] * (N - 1))


def dynamics(state, t, p):
    # state = [x_0, ..., x_{N-1}, v_0, ..., v_{N-1}]
    x, v = jnp.split(state, 2)
    k_springs, beta = unpack_params(p)

    # Fixed damping coefficient (not optimizable)
    
    # fixed‐end springs → pad x with zeros at both ends
    x_pad = jnp.concatenate([jnp.array([0.0]), x, jnp.array([0.0])])
    
    # linear spring forces with individual spring constants (vectorized)
    # For mass i: force = -k[i]*(x[i] - x[i-1]) - k[i+1]*(x[i] - x[i+1])
    left_neighbors = x_pad[0:N]      # [0, x[0], x[1], ..., x[N-2]]
    right_neighbors = x_pad[2:N+2]   # [x[1], x[2], ..., x[N-1], 0]
    left_springs = k_springs[0:N]    # spring constants for left connections
    right_springs = k_springs[1:N+1] # spring constants for right connections
    
    Kx = left_springs * (left_neighbors - x) + right_springs * (right_neighbors - x)
    
    # local Duffing nonlinearity
    duff = -beta * x**3
    
    # damping force (proportional to velocity)
    damping = -gamma * v
    
    # acceleration: F = ma → a = (spring + nonlinear + damping + external) / m
    a = ( Kx + duff  + damping + forcing(t) ) / mass

    return jnp.concatenate([v, a])

def solve_dynamics_free(p, init_state, ts):
    return odeint(dynamics, init_state, ts, p, rtol=rtol, atol=atol)



# Run "true" model and define initial model for learning

In [ ]:
# Cell 3: Create a synthetic “true” trajectory
# Parameters: [k_0, k_1, ..., k_N, beta_0, beta_1, ..., beta_{N-1}]
# N+1 spring constants + N Duffing coefficients
p_true = jnp.array([1.0] * (N+1) + [0.5] * N)   # true parameters
p_initial = p_true * 1.5
print(f"Parameter vector length: {len(p_true)} (expected: {2*N+1})")
print(f"Spring constants: {p_true[:N+1]}")
print(f"Duffing coefficients: {p_true[N+1:]}")
t_final = 10.0
num_steps = 200
ts = jnp.linspace(0.0, t_final, num_steps)
init_state = jnp.zeros(2 * N)


# forward integrate to get target
target_states = solve_dynamics_free(p_true, init_state, ts)
target_traj  = target_states[:, N-1]    # record last mass’s position
target_traj_vel = target_states[:, 2*N-1]
print("Target trajectory shape:", target_traj.shape)
print("Target trajectory velocity shape:", target_traj_vel.shape)

## Plot true model

In [ ]:
# Cell 5: Plot the final DOF over time
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(ts, target_traj, 'b-', linewidth=2, label='Final mass position')
plt.xlabel('Time')
plt.ylabel('Position')
plt.title('Final Degree of Freedom (Last Mass) Over Time')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Final mass trajectory range: [{target_traj.min():.4f}, {target_traj.max():.4f}]")


# Define clamped dynamics

In [ ]:
@jax.jit
def dynamics_clamped(state, evolution_clamped_dof_pos, t, p):
    """
    Clamped dynamics where the measurement DOF (final mass) position and velocity 
    exactly follow the target trajectory, while all accelerations are determined 
    by natural dynamics. This is a displacement and forcing control problem.
    
    The constraint forces emerge naturally from clamping displacement and velocity.
    """
    # state = [x_0, ..., x_{N-1}, v_0, ..., v_{N-1}]
    x, v = jnp.split(state, 2)
    k_springs, beta = unpack_params(p)
    

    # Clamp the final mass position to exactly match the target trajectory
    x_clamped = x.at[N-1].set(jnp.interp(t, ts, evolution_clamped_dof_pos))
    
    # Clamp the final mass velocity to match target trajectory derivative
    # Use JAX-compatible gradient computation to avoid boolean conversion errors
    v_clamped = v

    
    # Compute accelerations for all masses using normal dynamics
    # fixed‐end springs → pad x with zeros at both ends
    x_pad = jnp.concatenate([jnp.array([0.0]), x_clamped, jnp.array([0.0])])
    
    # linear spring forces with individual spring constants (vectorized)
    # For mass i: force = -k[i]*(x[i] - x[i-1]) - k[i+1]*(x[i] - x[i+1])
    left_neighbors = x_pad[0:N]      # [0, x[0], x[1], ..., x[N-2]]
    right_neighbors = x_pad[2:N+2]   # [x[1], x[2], ..., x[N-1], 0]
    left_springs = k_springs[0:N]    # spring constants for left connections
    right_springs = k_springs[1:N+1] # spring constants for right connections
    
    Kx = left_springs * (left_neighbors - x_clamped) + right_springs * (right_neighbors - x_clamped)
    
    # local Duffing nonlinearity
    duff = -beta * x_clamped**3
    
    # damping force (proportional to velocity)
    damping = - gamma * v_clamped
    
    # acceleration: F = ma → a = (spring + nonlinear + damping + external) / m
    a = (Kx + duff  + damping + forcing(t)) / mass
    
    # Let natural dynamics determine all accelerations (including final mass)
    # The displacement and velocity clamping will create the constraint forces

    return jnp.concatenate([v_clamped, a])

def solve_clamped_dynamics(p, clamped_dof_pos, init_state, ts):
    _dynamics_clamped = lambda state, t, p: dynamics_clamped(state, clamped_dof_pos, t, p)
    return odeint(_dynamics_clamped, init_state, ts, p, rtol=rtol, atol=atol)
    

## Test clamped dynamics

In [ ]:
# Cell 9: Test the new clamped dynamics method

print("=== Testing Clamped Dynamics Training Method ===")

# Test with different parameters to see how the system responds

# Integrate the clamped dynamics
states_clamped = solve_clamped_dynamics(p_true, target_traj, init_state, ts)

# Extract trajectories
x_clamped = states_clamped[:, :N]  # positions of all masses
v_clamped = states_clamped[:, N:]  # velocities of all masses

# Compare with original dynamics and target
states_original = target_states
x_original = states_original[:, :N]

print("Clamped dynamics integration completed successfully!")
print(f"Final mass clamping error: {jnp.max(jnp.abs(x_clamped[:, N-1] - target_traj)):.2e}")

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Plot 1: Position trajectories comparison
ax = axes[0, 0]
for i in range(N):
    if i == N-1:  # final mass (measurement DOF)
        ax.plot(ts, target_traj, 'k-', linewidth=3, alpha=0.8, label=f'Target (mass {i})')
        ax.plot(ts, x_clamped[:, i], 'r--', linewidth=2, alpha=0.7, label=f'Clamped (mass {i})')
        ax.plot(ts, x_original[:, i], 'b:', linewidth=2, alpha=0.7, label=f'Original (mass {i})')
    else:
        ax.plot(ts, x_clamped[:, i], 'r-', linewidth=1.5, alpha=0.7, label=f'Clamped (mass {i})')
        ax.plot(ts, x_original[:, i], 'b--', linewidth=1, alpha=0.5, label=f'Original (mass {i})')

ax.set_xlabel('Time')
ax.set_ylabel('Position')
ax.set_title('Position Trajectories: Clamped vs Original Dynamics')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

# Plot 2: Focus on measurement DOF
ax = axes[0, 1]
ax.plot(ts, target_traj, 'k-', linewidth=3, label='Target trajectory')
ax.plot(ts, x_clamped[:, N-1], 'r--', linewidth=2, label='Clamped dynamics')
ax.plot(ts, x_original[:, N-1], 'b:', linewidth=2, label='Original dynamics')
ax.set_xlabel('Time')
ax.set_ylabel('Position')
ax.set_title(f'Measurement DOF (Mass {N-1}): Target vs Dynamics')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Velocity comparison of final mass
ax = axes[0, 2]
target_vel_computed = jnp.gradient(target_traj, ts)  # Target velocity from gradient
ax.plot(ts, target_traj_vel, 'k-', linewidth=3, label='Target velocity (original)')
ax.plot(ts, target_vel_computed, 'k:', linewidth=2, label='Target velocity (gradient)', alpha=0.7)
ax.plot(ts, v_clamped[:, N-1], 'r--', linewidth=2, label='Clamped velocity')
ax.plot(ts, states_original[:, N+N-1], 'b:', linewidth=2, label='Original velocity')
ax.set_xlabel('Time')
ax.set_ylabel('Velocity')
ax.set_title(f'Velocity Comparison (Mass {N-1})')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Phase space of final mass
ax = axes[1, 0]
ax.plot(x_clamped[:, N-1], v_clamped[:, N-1], 'r-', linewidth=2, label='Clamped dynamics')
ax.plot(x_original[:, N-1], states_original[:, N+N-1], 'b--', linewidth=2, label='Original dynamics')
ax.plot(target_traj, jnp.gradient(target_traj, ts), 'k-', linewidth=3, alpha=0.7, label='Target')
ax.set_xlabel('Position (final mass)')
ax.set_ylabel('Velocity (final mass)')
ax.set_title('Phase Space: Final Mass')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 5: Internal mass responses
ax = axes[1, 1]
for i in range(N-1):  # all masses except the final one
    ax.plot(ts, x_clamped[:, i] - x_original[:, i], 
            linewidth=2, label=f'Mass {i} difference')
ax.set_xlabel('Time')
ax.set_ylabel('Position difference (Clamped - Original)')
ax.set_title('How Internal Masses Respond to Output Constraint')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 6: Velocity error analysis
ax = axes[1, 2]
vel_error_clamped = v_clamped[:, N-1] - target_traj_vel
vel_error_gradient = v_clamped[:, N-1] - target_vel_computed
ax.plot(ts, vel_error_clamped, 'r-', linewidth=2, label='Clamped - Target (original)')
ax.plot(ts, vel_error_gradient, 'g--', linewidth=2, label='Clamped - Target (gradient)')
ax.plot(ts, jnp.zeros_like(ts), 'k:', alpha=0.5, label='Zero error')
ax.set_xlabel('Time')
ax.set_ylabel('Velocity Error')
ax.set_title('Velocity Clamping Errors')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Analysis ===")
print(f"Maximum clamping error: {jnp.max(jnp.abs(x_clamped[:, N-1] - target_traj)):.2e}")
print(f"RMS difference in internal masses:")
for i in range(N-1):
    rms_diff = jnp.sqrt(jnp.mean((x_clamped[:, i] - x_original[:, i])**2))
    print(f"  Mass {i}: {rms_diff:.4f}")
    
print(f"\nThis demonstrates the new training method where:")
print(f"1. The measurement DOF (mass {N-1}) exactly follows the target")
print(f"2. Internal masses adjust to satisfy this constraint")
print(f"3. The system provides information about parameter sensitivity")
print(f"4. Everything runs forward in time with applied forcing")


# Define force matching loss function with hidden DOFs

In [ ]:
@jax.jit
def _clamped_loss_fn(p, states_clamped, evolution_clamped_dofs):
    """
    Force matching loss function for clamped dynamics training.
    
    Strategy:
    1. Run clamped dynamics to get target [v_clamped; a_clamped] for ALL DOFs
    2. Run free dynamics to get [v_free; a_free] for ALL DOFs  
    3. Loss = ||[v_clamped; a_clamped] - [v_free; a_free]||^2
    
    If parameters are correct, free dynamics should naturally match clamped behavior.
    """
    # Get clamped trajectory (measurement DOF follows target exactly)
    
    
    # Get free trajectory (all DOFs evolve naturally)  
    states_free = solve_dynamics_free(p, init_state, ts)
    
    # Extract velocities for all DOFs
    v_clamped = states_clamped[:, N:]  # velocities from clamped dynamics
    v_free = states_free[:, N:]        # velocities from free dynamics
    
    
        # Compute accelerations for all DOFs by evaluating dynamics functions
    def get_accelerations(states, dynamics_fn):
        """Extract accelerations by evaluating dynamics function at each time point"""
        # Use vmap to vectorize over states and time arrays
        vectorized_dynamics = jax.vmap(lambda state, t: dynamics_fn(state, t, p))
        dstates_dt = vectorized_dynamics(states, ts)
        accelerations = dstates_dt[:, N:]  # accelerations are second half
        return accelerations
    
    a_clamped = get_accelerations(states_clamped, dynamics)
    a_free = get_accelerations(states_free, dynamics)
    
    
    # Force matching loss: compare velocities and accelerations
    v_diff = v_clamped - v_free
    a_diff = a_clamped - a_free
    
    # Combined velocity and acceleration matching
    velocity_loss = 0.5 * jnp.trapezoid(jnp.sum(v_diff**2, axis=1), ts)
    acceleration_loss = 0.5 * jnp.trapezoid(jnp.sum(a_diff**2, axis=1), ts)
    
    return velocity_loss + acceleration_loss



# Run "brute force" force matching

In [ ]:

print("=== Free Dynamics vs Target Trajectory Comparison ===")


states_clamped = solve_clamped_dynamics(p_initial, target_traj, init_state, ts)

clamped_loss_fn = lambda p: _clamped_loss_fn(p, states_clamped, target_traj)

# Create JIT-compiled gradient function for faster optimization
clamped_loss_grad = jax.jit(jax.grad(clamped_loss_fn))

# Run a few optimization steps with force matching
@jax.jit
def gd_step(p, grad, lr=0.0008):
    return p - lr * grad


# Perform optimization steps
p_current = p_initial.copy()
losses = [clamped_loss_fn(p_current)]

lr = 0.0008   
print("Performing optimization steps...")
for step in range(1000):
    grad = clamped_loss_grad(p_current)  # Use JIT-compiled gradient
    print(f"grad: {grad}")
    p_current = gd_step(p_current, grad, lr=lr)
    print(f"p_current: {p_current}")
    loss = clamped_loss_fn(p_current)
    losses.append(loss)
    if step % 1 == 0:
        print(f"  Step {step}: loss = {loss:.6f}")

print(f"Final step: loss = {losses[-1]:.6f}")

# Run final free dynamics after optimization
states_final = odeint(dynamics, init_state, ts, p_current, rtol=1e-12, atol=1e-12)

# Get target trajectory and its velocity
target_x = target_traj  # position of final mass
target_v = jnp.gradient(target_traj, ts)  # velocity of final mass


# Run initial free dynamics
states_initial = solve_dynamics_free(p_initial, init_state, ts)
# Extract final mass trajectories
x_initial_last = states_initial[:, N-1]   # initial free dynamics, last mass position
v_initial_last = states_initial[:, 2*N-1] # initial free dynamics, last mass velocity

x_final_last = states_final[:, N-1]       # final free dynamics, last mass position  
v_final_last = states_final[:, 2*N-1]     # final free dynamics, last mass velocity



## Plot "brute force" force matching results

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Position comparison
ax = axes[0, 0]
ax.plot(ts, target_x, 'k-', linewidth=3, label='Target trajectory', alpha=0.8)
ax.plot(ts, x_initial_last, 'r--', linewidth=2, label='Initial free dynamics', alpha=0.7)
ax.plot(ts, x_final_last, 'b-', linewidth=2, label='Optimized free dynamics', alpha=0.7)
ax.set_xlabel('Time')
ax.set_ylabel('Position')
ax.set_title('Final Mass Position: Target vs Free Dynamics')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Velocity comparison
ax = axes[0, 1]
ax.plot(ts, target_v, 'k-', linewidth=3, label='Target velocity', alpha=0.8)
ax.plot(ts, v_initial_last, 'r--', linewidth=2, label='Initial free dynamics', alpha=0.7)
ax.plot(ts, v_final_last, 'b-', linewidth=2, label='Optimized free dynamics', alpha=0.7)
ax.set_xlabel('Time')
ax.set_ylabel('Velocity')
ax.set_title('Final Mass Velocity: Target vs Free Dynamics')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Phase space comparison
ax = axes[1, 0]
ax.plot(target_x, target_v, 'k-', linewidth=3, label='Target', alpha=0.8)
ax.plot(x_initial_last, v_initial_last, 'r--', linewidth=2, label='Initial free', alpha=0.7)
ax.plot(x_final_last, v_final_last, 'b-', linewidth=2, label='Optimized free', alpha=0.7)

# Mark initial and final points
ax.scatter(target_x[0], target_v[0], color='black', s=100, marker='o', label='Start', zorder=5)
ax.scatter(target_x[-1], target_v[-1], color='black', s=100, marker='s', label='End', zorder=5)

ax.set_xlabel('Position')
ax.set_ylabel('Velocity')
ax.set_title('Phase Space: Final Mass')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Optimization convergence
ax = axes[1, 1]
ax.semilogy(range(len(losses)), losses, 'bo-', linewidth=2, markersize=4)
ax.set_xlabel('Optimization Step')
ax.set_ylabel('Force Matching Loss (log scale)')
ax.set_title('Training Convergence')
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save the plot
if save_flag:
    plt.savefig(f"{output_dir}/force_matching_training.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_dir}/force_matching_training.pdf", bbox_inches='tight')
    print(f"Saved plot: force_matching_training.png and .pdf")

plt.show()

# Quantitative comparison
print(f"\\n=== Quantitative Analysis ===")

# Position errors
pos_error_initial = jnp.sqrt(jnp.mean((x_initial_last - target_x)**2))
pos_error_final = jnp.sqrt(jnp.mean((x_final_last - target_x)**2))

print(f"Position RMS Error:")
print(f"  Initial: {pos_error_initial:.6f}")
print(f"  Final:   {pos_error_final:.6f}")
print(f"  Improvement: {pos_error_initial/pos_error_final:.2f}x")

# Velocity errors
vel_error_initial = jnp.sqrt(jnp.mean((v_initial_last - target_v)**2))
vel_error_final = jnp.sqrt(jnp.mean((v_final_last - target_v)**2))

print(f"\\nVelocity RMS Error:")
print(f"  Initial: {vel_error_initial:.6f}")
print(f"  Final:   {vel_error_final:.6f}")
print(f"  Improvement: {vel_error_initial/vel_error_final:.2f}x")

# Parameter recovery
param_error_initial = jnp.linalg.norm(p_initial - p_true)
param_error_final = jnp.linalg.norm(p_current - p_true)

print(f"\\nParameter Recovery:")
print(f"  Initial ||p - p_true||: {param_error_initial:.6f}")
print(f"  Final ||p - p_true||:   {param_error_final:.6f}")
print(f"  Improvement: {param_error_initial/param_error_final:.2f}x")

print(f"\\nLoss Reduction:")
print(f"  Initial loss: {losses[0]:.8f}")
print(f"  Final loss:   {losses[-1]:.8f}")
print(f"  Reduction: {losses[0]/losses[-1]:.2f}x")

print(f"\\n=== Demonstration Complete ===")
print("This shows how force matching training drives the free dynamics")
print("to naturally match the target trajectory without explicit constraints!")


# Gentle force matching

## Interpolating between free and target trajectory:

In [ ]:
# Reset to initial (wrong) parameters

states_free_initial = solve_dynamics_free(p_initial, init_state, ts)
x_free_last = states_free_initial[:, N-1]

# Interpolate between free and target trajectories
x_target_alpha = lambda alpha: (1 - alpha) * x_free_last + alpha * target_traj

n_alpha_levels = 15  # Number of alpha levels to use

alpha_values = jnp.linspace(0, 1, n_alpha_levels)

# Plot trajectories for different alpha values
plt.figure(figsize=(12, 8))

# Plot a selection of alpha values to avoid overcrowding
alpha_plot_values = alpha_values
colors = plt.cm.viridis(jnp.linspace(0, 1, len(alpha_plot_values)))

for i, alpha in enumerate(alpha_plot_values):
    x_interp = x_target_alpha(alpha)
    plt.plot(ts, x_interp, color=colors[i], linewidth=2, 
             label=f'α = {alpha:.1f}', alpha=0.8)

# Highlight the endpoints
plt.plot(ts, x_free_last, 'r--', linewidth=3, 
         label='Free trajectory (α=0)', alpha=0.9)
plt.plot(ts, target_traj, 'k-', linewidth=3, 
         label='Target trajectory (α=1)', alpha=0.9)

plt.xlabel('Time')
plt.ylabel('Position (Final Mass)')
plt.title('Interpolated Trajectories Between Free and Target\n(Gentle Force Matching Setup)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Plotted trajectories for α ∈ {alpha_plot_values}")
print(f"Free trajectory final value: {x_free_last[-1]:.4f}")
print(f"Target trajectory final value: {target_traj[-1]:.4f}")
print(f"Total alpha values available: {len(alpha_values)}")


## Run Gentle Force Matching

In [ ]:
# CLEAN OPTIMIZATION LOOP: Using Clean Gentle Force Matching
print("=== CLEAN OPTIMIZATION LOOP: Using Clean Gentle Force Matching ===")

# Gentle force matching schedule parameters
steps_per_alpha = 5000  # Steps per alpha level
total_steps = n_alpha_levels * steps_per_alpha
lr = 1e-3  # Learning rate

print(f"CLEAN Gentle Force Matching Schedule:")
print(f"  Alpha levels: {n_alpha_levels}")
print(f"  Steps per alpha: {steps_per_alpha}")
print(f"  Total steps: {total_steps}")
print(f"  Learning rate: {lr}")
print(f"  Method: CLEAN clamped dynamics using existing _clamped_loss_fn")
print()

# Track optimization progress
gentle_losses_clean = []
alphas_clean = []
param_errors_clean = []
trajectory_errors_clean = []
alpha_transitions_clean = []  # Track when alpha changes

step_count = 0

p_current = p_initial
for alpha_idx in range(n_alpha_levels):
    # Compute current alpha (gradually increase from 0 to 1)
    alpha_current = alpha_values[alpha_idx]
    
    _x_target_alpha = x_target_alpha(alpha_current)
    states_clamped_alpha = solve_clamped_dynamics(p_current, _x_target_alpha, init_state, ts)
    
    clamped_loss_fn = lambda p: _clamped_loss_fn(p, states_clamped_alpha, _x_target_alpha)

    # Create JIT-compiled gradient function for faster optimization
    clamped_loss_grad = jax.jit(jax.grad(clamped_loss_fn))
    
    print(f"Alpha level: α = {alpha_current:.3f}")
    
    # Run multiple optimization steps at this alpha level
    for step_in_alpha in range(steps_per_alpha):

        
        # Compute gradient and update parameters using CLEAN functions
        grad = clamped_loss_grad(p_current)
        p_current = p_current - lr * grad
        
        # Compute loss using CLEAN function
        loss = clamped_loss_fn(p_current)
        gentle_losses_clean.append(loss)
        alphas_clean.append(alpha)
        
        # Track parameter and trajectory errors
        param_error = jnp.linalg.norm(p_current - p_true) / jnp.linalg.norm(p_true)
        param_errors_clean.append(param_error)
        
        # Get current trajectory error
        states_current = odeint(dynamics, init_state, ts, p_current, rtol=1e-12, atol=1e-12)
        x_current_last = states_current[:, N-1]
        traj_error = jnp.linalg.norm(x_current_last - target_traj) / jnp.linalg.norm(target_traj)
        trajectory_errors_clean.append(traj_error)
        
        step_count += 1
        
        # Print progress for first/last few steps at each alpha
        if step_in_alpha < 3 or step_in_alpha >= steps_per_alpha - 3:
            print(f"    Step {step_in_alpha+1:2d}/{steps_per_alpha}: loss={loss:.6f}, param_err={param_error:.4f}, traj_err={traj_error:.4f}")
    
    # Mark alpha transition
    alpha_transitions_clean.append(step_count - 1)
    
    # Print summary for this alpha level
    final_loss_at_alpha = gentle_losses_clean[-1]
    final_param_err_at_alpha = param_errors_clean[-1]
    final_traj_err_at_alpha = trajectory_errors_clean[-1]
    print(f"  → Final at α={alpha:.3f}: loss={final_loss_at_alpha:.6f}, param_err={final_param_err_at_alpha:.4f}, traj_err={final_traj_err_at_alpha:.4f}")
    print()

print(f"=== FINAL RESULTS (CLEAN IMPLEMENTATION) ===")
print(f"  Final loss: {gentle_losses_clean[-1]:.6f}")
print(f"  Final parameter error: {param_errors_clean[-1]:.4f}")
print(f"  Final trajectory error: {trajectory_errors_clean[-1]:.4f}")
print(f"  True parameters: {p_true}")
print(f"  Final parameters: {p_current}")
print(f"\\n🎉 CLEAN Gentle Force Matching completed successfully!")
print(f"Progressed through {n_alpha_levels} alpha levels with {steps_per_alpha} steps each.")
print()
print(f"🔧 Key Features:")
print(f"✓ CLEAN implementation using existing _clamped_loss_fn")
print(f"✓ Alpha-interpolated trajectories: x_alpha = (1-α)*x_free + α*x_target")
print(f"✓ Clamped dynamics: final mass follows interpolated trajectory exactly")
print(f"✓ No code duplication: reuses existing force matching logic")
print(f"✓ Gentle progression: smooth transition from free (α=0) to target (α=1)")


## Analyize gentlre force matching results

In [ ]:
# Visualization of Clean Gentle Force Matching Progress
print("=== Visualizing Clean Gentle Force Matching Progress ===")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Convert to numpy arrays for plotting
gentle_losses_np = jnp.array(gentle_losses_clean)
alphas_np = jnp.array(alphas_clean)
param_errors_np = jnp.array(param_errors_clean)
trajectory_errors_np = jnp.array(trajectory_errors_clean)
steps = jnp.arange(len(gentle_losses_clean))

# Plot 1: Loss vs Steps with Alpha Schedule
ax1 = axes[0, 0]
ax1.plot(steps, gentle_losses_np, 'b-', linewidth=2, label='Loss')
ax1.set_xlabel('Optimization Step')
ax1.set_ylabel('Loss', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_title('Loss Evolution During Clean Gentle Force Matching')
ax1.grid(True, alpha=0.3)

# Add alpha schedule on secondary y-axis
ax1_twin = ax1.twinx()
ax1_twin.plot(steps, alphas_np, 'r--', linewidth=2, alpha=0.7, label='Alpha')
ax1_twin.set_ylabel('Alpha (Interpolation Factor)', color='r')
ax1_twin.tick_params(axis='y', labelcolor='r')

# Add vertical lines at alpha transitions
for transition in alpha_transitions_clean:
    ax1.axvline(x=transition, color='gray', linestyle=':', alpha=0.5)

# Plot 2: Parameter Error vs Steps
ax2 = axes[0, 1]
ax2.semilogy(steps, param_errors_np, 'g-', linewidth=2)
ax2.set_xlabel('Optimization Step')
ax2.set_ylabel('Parameter Error (log scale)')
ax2.set_title('Parameter Convergence')
ax2.grid(True, alpha=0.3)

# Add vertical lines at alpha transitions
for transition in alpha_transitions_clean:
    ax2.axvline(x=transition, color='gray', linestyle=':', alpha=0.5)

# Plot 3: Trajectory Error vs Steps
ax3 = axes[1, 0]
ax3.semilogy(steps, trajectory_errors_np, 'm-', linewidth=2)
ax3.set_xlabel('Optimization Step')
ax3.set_ylabel('Trajectory Error (log scale)')
ax3.set_title('Trajectory Matching Progress')
ax3.grid(True, alpha=0.3)

# Add vertical lines at alpha transitions
for transition in alpha_transitions_clean:
    ax3.axvline(x=transition, color='gray', linestyle=':', alpha=0.5)

# Plot 4: Loss vs Alpha (showing convergence at each alpha level)
ax4 = axes[1, 1]
scatter = ax4.scatter(alphas_np, gentle_losses_np, c=steps, cmap='viridis', alpha=0.6, s=10)
ax4.set_xlabel('Alpha (Interpolation Factor)')
ax4.set_ylabel('Loss')
ax4.set_title('Loss vs Alpha (Color = Step Number)')
ax4.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax4, label='Step Number')

plt.tight_layout()

# Save the plot
if save_flag:
    plt.savefig(f"{output_dir}/interpolated_trajectories.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_dir}/interpolated_trajectories.pdf", bbox_inches='tight')
    print(f"Saved plot: interpolated_trajectories.png and .pdf")

plt.show()

# Print summary statistics
print(f"\\n=== Clean Gentle Force Matching Summary ===")
print(f"Initial parameter error: {param_errors_np[0]:.4f}")
print(f"Final parameter error: {param_errors_np[-1]:.4f}")
print(f"Parameter error reduction: {param_errors_np[0]/param_errors_np[-1]:.1f}x")
print()
print(f"Initial trajectory error: {trajectory_errors_np[0]:.4f}")
print(f"Final trajectory error: {trajectory_errors_np[-1]:.4f}")
print(f"Trajectory error reduction: {trajectory_errors_np[0]/trajectory_errors_np[-1]:.1f}x")
print()
print(f"Initial loss: {gentle_losses_np[0]:.6f}")
print(f"Final loss: {gentle_losses_np[-1]:.6f}")
print(f"Loss reduction: {gentle_losses_np[0]/gentle_losses_np[-1]:.1f}x")


In [ ]:
# Trajectory Comparison: Final Results vs Target
print("=== Trajectory Comparison: Final Results vs Target ===")

# Generate trajectories with different parameter sets for comparison
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Get initial (wrong) trajectory
states_initial = odeint(dynamics, init_state, ts, p_initial, rtol=1e-12, atol=1e-12)
x_initial = states_initial[:, N-1]  # Final mass trajectory
v_initial = states_initial[:, 2*N-1]  # Final mass velocity

# 2. Get final optimized trajectory
states_final = odeint(dynamics, init_state, ts, p_current, rtol=1e-12, atol=1e-12)
x_final = states_final[:, N-1]  # Final mass trajectory
v_final = states_final[:, 2*N-1]  # Final mass velocity

# 3. Get target velocity for comparison
target_v = jnp.gradient(target_traj, ts)

# Plot 1: Position trajectories
ax1 = axes[0, 0]
ax1.plot(ts, target_traj, 'k-', linewidth=3, label='Target Trajectory', alpha=0.9)
ax1.plot(ts, x_initial, 'r--', linewidth=2, label='Initial (30% off params)', alpha=0.7)
ax1.plot(ts, x_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('Position')
ax1.set_title('Position Trajectories Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Velocity trajectories
ax2 = axes[0, 1]
ax2.plot(ts, target_v, 'k-', linewidth=3, label='Target Velocity', alpha=0.9)
ax2.plot(ts, v_initial, 'r--', linewidth=2, label='Initial (30% off params)', alpha=0.7)
ax2.plot(ts, v_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
ax2.set_xlabel('Time')
ax2.set_ylabel('Velocity')
ax2.set_title('Velocity Trajectories Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Phase space (position vs velocity)
ax3 = axes[0, 2]
ax3.plot(target_traj, target_v, 'k-', linewidth=3, label='Target', alpha=0.9)
ax3.plot(x_initial, v_initial, 'r--', linewidth=2, label='Initial', alpha=0.7)
ax3.plot(x_final, v_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
# Mark start and end points
ax3.scatter(target_traj[0], target_v[0], color='black', s=100, marker='o', label='Start', zorder=5)
ax3.scatter(target_traj[-1], target_v[-1], color='black', s=100, marker='s', label='End', zorder=5)
ax3.set_xlabel('Position')
ax3.set_ylabel('Velocity')
ax3.set_title('Phase Space Trajectories')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Position error over time
ax4 = axes[1, 0]
pos_error_initial = jnp.abs(x_initial - target_traj)
pos_error_final = jnp.abs(x_final - target_traj)
ax4.semilogy(ts, pos_error_initial, 'r--', linewidth=2, label='Initial Error', alpha=0.7)
ax4.semilogy(ts, pos_error_final, 'b-', linewidth=2, label='Final Error', alpha=0.8)
ax4.set_xlabel('Time')
ax4.set_ylabel('Position Error (log scale)')
ax4.set_title('Position Error Over Time')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Plot 5: Velocity error over time
ax5 = axes[1, 1]
vel_error_initial = jnp.abs(v_initial - target_v)
vel_error_final = jnp.abs(v_final - target_v)
ax5.semilogy(ts, vel_error_initial, 'r--', linewidth=2, label='Initial Error', alpha=0.7)
ax5.semilogy(ts, vel_error_final, 'b-', linewidth=2, label='Final Error', alpha=0.8)
ax5.set_xlabel('Time')
ax5.set_ylabel('Velocity Error (log scale)')
ax5.set_title('Velocity Error Over Time')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Parameter comparison
ax6 = axes[1, 2]
param_names = [f'p[{i}]' for i in range(len(p_true))]
x_pos = jnp.arange(len(p_true))
width = 0.25

ax6.bar(x_pos - width, p_true, width, label='True Parameters', alpha=0.8, color='black')
ax6.bar(x_pos, p_initial, width, label='Initial Parameters', alpha=0.7, color='red')
ax6.bar(x_pos + width, p_current, width, label='Final Optimized', alpha=0.8, color='blue')

ax6.set_xlabel('Parameter Index')
ax6.set_ylabel('Parameter Value')
ax6.set_title('Parameter Comparison')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(param_names)
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()

# Save the plot
if save_flag:
    plt.savefig(f"{output_dir}/clean_gentle_force_matching_progress.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_dir}/clean_gentle_force_matching_progress.pdf", bbox_inches='tight')
    print(f"Saved plot: clean_gentle_force_matching_progress.png and .pdf")

plt.show()

# Print quantitative comparison
print(f"\\n=== Quantitative Trajectory Comparison ===")

# Position errors
pos_rmse_initial = jnp.sqrt(jnp.mean((x_initial - target_traj)**2))
pos_rmse_final = jnp.sqrt(jnp.mean((x_final - target_traj)**2))
pos_max_error_initial = jnp.max(jnp.abs(x_initial - target_traj))
pos_max_error_final = jnp.max(jnp.abs(x_final - target_traj))

print(f"Position RMSE:")
print(f"  Initial: {pos_rmse_initial:.6f}")
print(f"  Final:   {pos_rmse_final:.6f}")
print(f"  Improvement: {pos_rmse_initial/pos_rmse_final:.1f}x")

print(f"\\nPosition Max Error:")
print(f"  Initial: {pos_max_error_initial:.6f}")
print(f"  Final:   {pos_max_error_final:.6f}")
print(f"  Improvement: {pos_max_error_initial/pos_max_error_final:.1f}x")

# Velocity errors
vel_rmse_initial = jnp.sqrt(jnp.mean((v_initial - target_v)**2))
vel_rmse_final = jnp.sqrt(jnp.mean((v_final - target_v)**2))
vel_max_error_initial = jnp.max(jnp.abs(v_initial - target_v))
vel_max_error_final = jnp.max(jnp.abs(v_final - target_v))

print(f"\\nVelocity RMSE:")
print(f"  Initial: {vel_rmse_initial:.6f}")
print(f"  Final:   {vel_rmse_final:.6f}")
print(f"  Improvement: {vel_rmse_initial/vel_rmse_final:.1f}x")

print(f"\\nVelocity Max Error:")
print(f"  Initial: {vel_max_error_initial:.6f}")
print(f"  Final:   {vel_max_error_final:.6f}")
print(f"  Improvement: {vel_max_error_initial/vel_max_error_final:.1f}x")

# Parameter errors
param_rmse_initial = jnp.sqrt(jnp.mean((p_initial - p_true)**2))
param_rmse_final = jnp.sqrt(jnp.mean((p_current - p_true)**2))

print(f"\\nParameter RMSE:")
print(f"  Initial: {param_rmse_initial:.6f}")
print(f"  Final:   {param_rmse_final:.6f}")
print(f"  Improvement: {param_rmse_initial/param_rmse_final:.1f}x")

print(f"\\n=== Success Metrics ===")
print(f"Position accuracy: {(1 - pos_rmse_final/pos_rmse_initial)*100:.1f}% improvement")
print(f"Velocity accuracy: {(1 - vel_rmse_final/vel_rmse_initial)*100:.1f}% improvement")
print(f"Parameter accuracy: {(1 - param_rmse_final/param_rmse_initial)*100:.1f}% improvement")


In [ ]:
# Enhanced Trajectory Comparison: Including Forces and Accelerations
print("=== Enhanced Trajectory Comparison: Final Results vs Target ===")

# Generate trajectories with different parameter sets for comparison
fig, axes = plt.subplots(3, 3, figsize=(24, 18))

# 1. Get initial (wrong) trajectory
states_initial = odeint(dynamics, init_state, ts, p_initial, rtol=1e-12, atol=1e-12)
x_initial = states_initial[:, N-1]  # Final mass trajectory
v_initial = states_initial[:, 2*N-1]  # Final mass velocity

# 2. Get final optimized trajectory
states_final = odeint(dynamics, init_state, ts, p_current, rtol=1e-12, atol=1e-12)
x_final = states_final[:, N-1]  # Final mass trajectory
v_final = states_final[:, 2*N-1]  # Final mass velocity

# 3. Get target velocity and acceleration
target_v = jnp.gradient(target_traj, ts)
target_a = jnp.gradient(target_v, ts)

# 4. Compute accelerations for initial and final trajectories
a_initial = jnp.gradient(v_initial, ts)
a_final = jnp.gradient(v_final, ts)

# 5. Compute forces (F = m * a, assuming unit mass)
target_force = target_a
force_initial = a_initial
force_final = a_final

# Row 1: Position, Velocity, Acceleration
# Plot 1: Position trajectories
ax1 = axes[0, 0]
ax1.plot(ts, target_traj, 'k-', linewidth=3, label='Target Trajectory', alpha=0.9)
ax1.plot(ts, x_initial, 'r--', linewidth=2, label='Initial (30% off params)', alpha=0.7)
ax1.plot(ts, x_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('Position')
ax1.set_title('Position Trajectories Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Velocity trajectories
ax2 = axes[0, 1]
ax2.plot(ts, target_v, 'k-', linewidth=3, label='Target Velocity', alpha=0.9)
ax2.plot(ts, v_initial, 'r--', linewidth=2, label='Initial (30% off params)', alpha=0.7)
ax2.plot(ts, v_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
ax2.set_xlabel('Time')
ax2.set_ylabel('Velocity')
ax2.set_title('Velocity Trajectories Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Acceleration trajectories
ax3 = axes[0, 2]
ax3.plot(ts, target_a, 'k-', linewidth=3, label='Target Acceleration', alpha=0.9)
ax3.plot(ts, a_initial, 'r--', linewidth=2, label='Initial (30% off params)', alpha=0.7)
ax3.plot(ts, a_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
ax3.set_xlabel('Time')
ax3.set_ylabel('Acceleration')
ax3.set_title('Acceleration Trajectories Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Row 2: Errors
# Plot 4: Position error over time
ax4 = axes[1, 0]
pos_error_initial = jnp.abs(x_initial - target_traj)
pos_error_final = jnp.abs(x_final - target_traj)
ax4.semilogy(ts, pos_error_initial, 'r--', linewidth=2, label='Initial Error', alpha=0.7)
ax4.semilogy(ts, pos_error_final, 'b-', linewidth=2, label='Final Error', alpha=0.8)
ax4.set_xlabel('Time')
ax4.set_ylabel('Position Error (log scale)')
ax4.set_title('Position Error Over Time')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Plot 5: Velocity error over time
ax5 = axes[1, 1]
vel_error_initial = jnp.abs(v_initial - target_v)
vel_error_final = jnp.abs(v_final - target_v)
ax5.semilogy(ts, vel_error_initial, 'r--', linewidth=2, label='Initial Error', alpha=0.7)
ax5.semilogy(ts, vel_error_final, 'b-', linewidth=2, label='Final Error', alpha=0.8)
ax5.set_xlabel('Time')
ax5.set_ylabel('Velocity Error (log scale)')
ax5.set_title('Velocity Error Over Time')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Acceleration/Force error over time
ax6 = axes[1, 2]
acc_error_initial = jnp.abs(a_initial - target_a)
acc_error_final = jnp.abs(a_final - target_a)
ax6.semilogy(ts, acc_error_initial, 'r--', linewidth=2, label='Initial Error', alpha=0.7)
ax6.semilogy(ts, acc_error_final, 'b-', linewidth=2, label='Final Error', alpha=0.8)
ax6.set_xlabel('Time')
ax6.set_ylabel('Acceleration Error (log scale)')
ax6.set_title('Acceleration/Force Error Over Time')
ax6.legend()
ax6.grid(True, alpha=0.3)

# Row 3: Phase space and comparisons
# Plot 7: Phase space (position vs velocity)
ax7 = axes[2, 0]
ax7.plot(target_traj, target_v, 'k-', linewidth=3, label='Target', alpha=0.9)
ax7.plot(x_initial, v_initial, 'r--', linewidth=2, label='Initial', alpha=0.7)
ax7.plot(x_final, v_final, 'b-', linewidth=2, label='Final Optimized', alpha=0.8)
# Mark start and end points
ax7.scatter(target_traj[0], target_v[0], color='black', s=100, marker='o', label='Start', zorder=5)
ax7.scatter(target_traj[-1], target_v[-1], color='black', s=100, marker='s', label='End', zorder=5)
ax7.set_xlabel('Position')
ax7.set_ylabel('Velocity')
ax7.set_title('Phase Space Trajectories')
ax7.legend()
ax7.grid(True, alpha=0.3)

# Plot 8: Force comparison (F = ma)
ax8 = axes[2, 1]
ax8.plot(ts, target_force, 'k-', linewidth=3, label='Target Force', alpha=0.9)
ax8.plot(ts, force_initial, 'r--', linewidth=2, label='Initial Force', alpha=0.7)
ax8.plot(ts, force_final, 'b-', linewidth=2, label='Final Optimized Force', alpha=0.8)
ax8.set_xlabel('Time')
ax8.set_ylabel('Force (F = ma)')
ax8.set_title('Force Comparison')
ax8.legend()
ax8.grid(True, alpha=0.3)

# Plot 9: Parameter comparison
ax9 = axes[2, 2]
param_names = [f'p[{i}]' for i in range(len(p_true))]
x_pos = jnp.arange(len(p_true))
width = 0.25

ax9.bar(x_pos - width, p_true, width, label='True Parameters', alpha=0.8, color='black')
ax9.bar(x_pos, p_initial, width, label='Initial Parameters', alpha=0.7, color='red')
ax9.bar(x_pos + width, p_current, width, label='Final Optimized', alpha=0.8, color='blue')

ax9.set_xlabel('Parameter Index')
ax9.set_ylabel('Parameter Value')
ax9.set_title('Parameter Comparison')
ax9.set_xticks(x_pos)
ax9.set_xticklabels(param_names)
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.tight_layout()

# Save the plot
if save_flag:
    plt.savefig(f"{output_dir}/enhanced_trajectory_comparison.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_dir}/enhanced_trajectory_comparison.pdf", bbox_inches='tight')
    print(f"Saved plot: enhanced_trajectory_comparison.png and .pdf")

plt.show()

# Enhanced quantitative comparison including forces
print(f"\\n=== Enhanced Quantitative Trajectory Comparison ===")

# Position errors
pos_rmse_initial = jnp.sqrt(jnp.mean((x_initial - target_traj)**2))
pos_rmse_final = jnp.sqrt(jnp.mean((x_final - target_traj)**2))
pos_max_error_initial = jnp.max(jnp.abs(x_initial - target_traj))
pos_max_error_final = jnp.max(jnp.abs(x_final - target_traj))

print(f"Position RMSE:")
print(f"  Initial: {pos_rmse_initial:.6f}")
print(f"  Final:   {pos_rmse_final:.6f}")
print(f"  Improvement: {pos_rmse_initial/pos_rmse_final:.1f}x")

# Velocity errors
vel_rmse_initial = jnp.sqrt(jnp.mean((v_initial - target_v)**2))
vel_rmse_final = jnp.sqrt(jnp.mean((v_final - target_v)**2))
vel_max_error_initial = jnp.max(jnp.abs(v_initial - target_v))
vel_max_error_final = jnp.max(jnp.abs(v_final - target_v))

print(f"\\nVelocity RMSE:")
print(f"  Initial: {vel_rmse_initial:.6f}")
print(f"  Final:   {vel_rmse_final:.6f}")
print(f"  Improvement: {vel_rmse_initial/vel_rmse_final:.1f}x")

# Acceleration/Force errors
acc_rmse_initial = jnp.sqrt(jnp.mean((a_initial - target_a)**2))
acc_rmse_final = jnp.sqrt(jnp.mean((a_final - target_a)**2))
acc_max_error_initial = jnp.max(jnp.abs(a_initial - target_a))
acc_max_error_final = jnp.max(jnp.abs(a_final - target_a))

print(f"\\nAcceleration/Force RMSE:")
print(f"  Initial: {acc_rmse_initial:.6f}")
print(f"  Final:   {acc_rmse_final:.6f}")
print(f"  Improvement: {acc_rmse_initial/acc_rmse_final:.1f}x")

print(f"\\nAcceleration/Force Max Error:")
print(f"  Initial: {acc_max_error_initial:.6f}")
print(f"  Final:   {acc_max_error_final:.6f}")
print(f"  Improvement: {acc_max_error_initial/acc_max_error_final:.1f}x")

# Parameter errors
param_rmse_initial = jnp.sqrt(jnp.mean((p_initial - p_true)**2))
param_rmse_final = jnp.sqrt(jnp.mean((p_current - p_true)**2))

print(f"\\nParameter RMSE:")
print(f"  Initial: {param_rmse_initial:.6f}")
print(f"  Final:   {param_rmse_final:.6f}")
print(f"  Improvement: {param_rmse_initial/param_rmse_final:.1f}x")

print(f"\\n=== Success Metrics ===")
print(f"Position accuracy: {(1 - pos_rmse_final/pos_rmse_initial)*100:.1f}% improvement")
print(f"Velocity accuracy: {(1 - vel_rmse_final/vel_rmse_initial)*100:.1f}% improvement")
print(f"Acceleration/Force accuracy: {(1 - acc_rmse_final/acc_rmse_initial)*100:.1f}% improvement")
print(f"Parameter accuracy: {(1 - param_rmse_final/param_rmse_initial)*100:.1f}% improvement")

print(f"\\n=== Force Matching Assessment ===")
print(f"This is the key metric for gentle force matching success!")
print(f"Force RMSE reduction: {acc_rmse_initial/acc_rmse_final:.1f}x")
print(f"Force matching quality: {(1 - acc_rmse_final/acc_rmse_initial)*100:.1f}% improvement")


# Traditional optimization

In [ ]:
# TRADITIONAL TRAJECTORY MATCHING: Direct Optimization with Automatic Differentiation
print("=== TRADITIONAL TRAJECTORY MATCHING: Direct Optimization ===")

def trajectory_matching_loss(p, target_traj):
    """
    Traditional trajectory matching loss function: 1/2 * ||u_target - u||^2
    
    This is the standard approach used in simulation-based optimization:
    1. Run forward simulation with parameters p
    2. Extract final mass trajectory u = x[:, N-1] 
    3. Compute squared error against target trajectory
    4. Use automatic differentiation to get gradients
    
    Args:
        p: current parameters
        target_traj: target trajectory for final mass
    
    Returns:
        loss: 1/2 * ||u_target - u||^2
    """
    # Forward simulation with current parameters
    states = solve_dynamics_free(p, init_state, ts)
    u_current = states[:, N-1]  # Extract final mass trajectory
    
    # Trajectory matching loss: 1/2 * ||u_target - u||^2
    trajectory_error = u_current - target_traj
    loss = 0.5 * jnp.sum(trajectory_error**2)
    
    return loss

# Create JIT-compiled versions for efficiency
trajectory_loss_fn = jax.jit(trajectory_matching_loss)
trajectory_loss_grad = jax.jit(jax.grad(trajectory_matching_loss))

print("Traditional trajectory matching implemented!")
print("- trajectory_loss_fn: JIT-compiled loss function")
print("- trajectory_loss_grad: JIT-compiled gradient function")
print()
print("🎯 Method:")
print("✓ Direct trajectory matching: loss = 1/2 * ||u_target - u||^2")
print("✓ Automatic differentiation through forward simulation")
print("✓ Standard approach for simulation-based parameter estimation")
print("✓ Requires differentiable simulator (not available in real experiments)")
print()


In [ ]:
# TRADITIONAL OPTIMIZATION LOOP: Direct Trajectory Matching
print("=== TRADITIONAL OPTIMIZATION LOOP: Direct Trajectory Matching ===")

# Optimization parameters
traditional_steps = 1000  # Total optimization steps
traditional_lr = 1e-2    # Learning rate (can be higher since no alpha scheduling)

print(f"Traditional Trajectory Matching Schedule:")
print(f"  Total steps: {traditional_steps}")
print(f"  Learning rate: {traditional_lr}")
print(f"  Method: Direct trajectory matching with automatic differentiation")
print()

# Track optimization progress
traditional_losses = []
traditional_param_errors = []
traditional_trajectory_errors = []

# Start with same initial parameters as gentle force matching
p_traditional = p_initial.copy()

print("Starting traditional optimization...")

for step in range(traditional_steps):
    # Compute gradient and update parameters
    grad = trajectory_loss_grad(p_traditional, target_traj)
    p_traditional = p_traditional - traditional_lr * grad
    
    # Compute loss
    loss = trajectory_loss_fn(p_traditional, target_traj)
    traditional_losses.append(loss)
    
    # Track parameter error
    param_error = jnp.linalg.norm(p_traditional - p_true) / jnp.linalg.norm(p_true)
    traditional_param_errors.append(param_error)
    
    # Track trajectory error (same as loss but normalized)
    states_current = odeint(dynamics, init_state, ts, p_traditional, rtol=1e-12, atol=1e-12)
    x_current_last = states_current[:, N-1]
    traj_error = jnp.linalg.norm(x_current_last - target_traj) / jnp.linalg.norm(target_traj)
    traditional_trajectory_errors.append(traj_error)
    
    # Print progress
    if step % 50 == 0 or step < 10 or step >= traditional_steps - 10:
        print(f"  Step {step:3d}/{traditional_steps}: loss={loss:.6f}, param_err={param_error:.4f}, traj_err={traj_error:.4f}")

print(f"\\n=== FINAL RESULTS (TRADITIONAL OPTIMIZATION) ===")
print(f"  Final loss: {traditional_losses[-1]:.6f}")
print(f"  Final parameter error: {traditional_param_errors[-1]:.4f}")
print(f"  Final trajectory error: {traditional_trajectory_errors[-1]:.4f}")
print(f"  True parameters: {p_true}")
print(f"  Final parameters: {p_traditional}")
print(f"\\n🎉 Traditional trajectory matching completed successfully!")
print(f"Completed {traditional_steps} optimization steps with direct gradient descent.")
print()
print(f"🔧 Key Features:")
print(f"✓ Direct trajectory matching: minimizes ||u_target - u||^2")
print(f"✓ Automatic differentiation through forward simulation")
print(f"✓ No alpha scheduling: direct optimization to target")
print(f"✓ Requires differentiable simulator (JAX/PyTorch)")
print(f"✓ Standard approach for simulation-based parameter estimation")


In [ ]:
# COMPARISON: Force Matching vs Traditional Optimization
print("=== COMPARISON: Force Matching vs Traditional Optimization ===")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Convert arrays for plotting
gentle_losses_np = jnp.array(gentle_losses_clean)
gentle_param_errors_np = jnp.array(param_errors_clean)
gentle_traj_errors_np = jnp.array(trajectory_errors_clean)
gentle_steps = jnp.arange(len(gentle_losses_clean))

traditional_losses_np = jnp.array(traditional_losses)
traditional_param_errors_np = jnp.array(traditional_param_errors)
traditional_traj_errors_np = jnp.array(traditional_trajectory_errors)
traditional_steps_np = jnp.arange(len(traditional_losses))

# Plot 1: Loss Comparison
ax1 = axes[0, 0]
ax1.semilogy(gentle_steps, gentle_losses_np, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax1.semilogy(traditional_steps_np, traditional_losses_np, 'r-', linewidth=2, label='Traditional', alpha=0.8)
ax1.set_xlabel('Optimization Step')
ax1.set_ylabel('Loss (log scale)')
ax1.set_title('Loss Evolution Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Add vertical lines for alpha transitions in force matching
for transition in alpha_transitions_clean:
    ax1.axvline(x=transition, color='blue', linestyle=':', alpha=0.3, linewidth=1)

# Plot 2: Parameter Error Comparison
ax2 = axes[0, 1]
ax2.semilogy(gentle_steps, gentle_param_errors_np, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax2.semilogy(traditional_steps_np, traditional_param_errors_np, 'r-', linewidth=2, label='Traditional', alpha=0.8)
ax2.set_xlabel('Optimization Step')
ax2.set_ylabel('Parameter Error (log scale)')
ax2.set_title('Parameter Convergence Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Add vertical lines for alpha transitions
for transition in alpha_transitions_clean:
    ax2.axvline(x=transition, color='blue', linestyle=':', alpha=0.3, linewidth=1)

# Plot 3: Trajectory Error Comparison
ax3 = axes[0, 2]
ax3.semilogy(gentle_steps, gentle_traj_errors_np, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax3.semilogy(traditional_steps_np, traditional_traj_errors_np, 'r-', linewidth=2, label='Traditional', alpha=0.8)
ax3.set_xlabel('Optimization Step')
ax3.set_ylabel('Trajectory Error (log scale)')
ax3.set_title('Trajectory Matching Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Add vertical lines for alpha transitions
for transition in alpha_transitions_clean:
    ax3.axvline(x=transition, color='blue', linestyle=':', alpha=0.3, linewidth=1)

# Plot 4: Final Trajectory Comparison
ax4 = axes[1, 0]
# Get final trajectories
states_gentle_final = odeint(dynamics, init_state, ts, p_current, rtol=1e-12, atol=1e-12)
x_gentle_final = states_gentle_final[:, N-1]

states_traditional_final = odeint(dynamics, init_state, ts, p_traditional, rtol=1e-12, atol=1e-12)
x_traditional_final = states_traditional_final[:, N-1]

ax4.plot(ts, target_traj, 'k-', linewidth=3, label='Target', alpha=0.9)
ax4.plot(ts, x_gentle_final, 'b--', linewidth=2, label='Force Matching', alpha=0.8)
ax4.plot(ts, x_traditional_final, 'r:', linewidth=2, label='Traditional', alpha=0.8)
ax4.set_xlabel('Time')
ax4.set_ylabel('Position')
ax4.set_title('Final Trajectory Comparison')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Plot 5: Parameter Recovery Comparison
ax5 = axes[1, 1]
param_names = [f'p[{i}]' for i in range(len(p_true))]
x_pos = jnp.arange(len(p_true))
width = 0.2

ax5.bar(x_pos - width, p_true, width, label='True Parameters', alpha=0.8, color='black')
ax5.bar(x_pos, p_current, width, label='Force Matching', alpha=0.8, color='blue')
ax5.bar(x_pos + width, p_traditional, width, label='Traditional', alpha=0.8, color='red')

ax5.set_xlabel('Parameter Index')
ax5.set_ylabel('Parameter Value')
ax5.set_title('Parameter Recovery Comparison')
ax5.set_xticks(x_pos)
ax5.set_xticklabels(param_names)
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Convergence Rate Analysis
ax6 = axes[1, 2]
# Plot convergence rate (improvement per step)
gentle_improvement = gentle_param_errors_np[0] / gentle_param_errors_np
traditional_improvement = traditional_param_errors_np[0] / traditional_param_errors_np

ax6.plot(gentle_steps, gentle_improvement, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax6.plot(traditional_steps_np, traditional_improvement, 'r-', linewidth=2, label='Traditional', alpha=0.8)
ax6.set_xlabel('Optimization Step')
ax6.set_ylabel('Parameter Error Improvement (ratio)')
ax6.set_title('Convergence Rate Comparison')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()

# Save the comparison plot
if save_flag:
    plt.savefig(f"{output_dir}/method_comparison.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_dir}/method_comparison.pdf", bbox_inches='tight')
    print(f"Saved plot: method_comparison.png and .pdf")

plt.show()


In [ ]:
# QUANTITATIVE COMPARISON SUMMARY
print("\\n=== QUANTITATIVE COMPARISON SUMMARY ===")

# Final metrics comparison
print("📊 FINAL PERFORMANCE COMPARISON:")
print(f"{'Metric':<25} {'Force Matching':<15} {'Traditional':<15} {'Winner':<10}")
print("-" * 70)

# Parameter error comparison
fm_param_error = param_errors_clean[-1]
trad_param_error = traditional_param_errors[-1]
param_winner = "Force Match" if fm_param_error < trad_param_error else "Traditional"
print(f"{'Parameter Error':<25} {fm_param_error:<15.6f} {trad_param_error:<15.6f} {param_winner:<10}")

# Trajectory error comparison
fm_traj_error = trajectory_errors_clean[-1]
trad_traj_error = traditional_trajectory_errors[-1]
traj_winner = "Force Match" if fm_traj_error < trad_traj_error else "Traditional"
print(f"{'Trajectory Error':<25} {fm_traj_error:<15.6f} {trad_traj_error:<15.6f} {traj_winner:<10}")

# Final loss comparison (note: different loss functions)
fm_final_loss = gentle_losses_clean[-1]
trad_final_loss = traditional_losses[-1]
print(f"{'Final Loss':<25} {fm_final_loss:<15.6f} {trad_final_loss:<15.6f} {'Different':<10}")

print()
print("🔍 CONVERGENCE ANALYSIS:")

# Convergence speed (steps to reach 90% of final improvement)
fm_target = 0.9 * (fm_param_error / param_errors_clean[0])
trad_target = 0.9 * (trad_param_error / traditional_param_errors[0])

fm_convergence_step = None
for i, err in enumerate(param_errors_clean):
    if err / param_errors_clean[0] <= fm_target:
        fm_convergence_step = i
        break

trad_convergence_step = None
for i, err in enumerate(traditional_param_errors):
    if err / traditional_param_errors[0] <= trad_target:
        trad_convergence_step = i
        break

if fm_convergence_step is not None and trad_convergence_step is not None:
    conv_winner = "Force Match" if fm_convergence_step < trad_convergence_step else "Traditional"
    print(f"Steps to 90% convergence: Force Matching = {fm_convergence_step}, Traditional = {trad_convergence_step}")
    print(f"Convergence speed winner: {conv_winner}")
else:
    print("Convergence analysis: One or both methods did not reach 90% convergence")

print()
print("🎯 METHOD CHARACTERISTICS:")
print("Force Matching (Gentle):")
print(f"  ✓ Total steps: {len(gentle_losses_clean)}")
print(f"  ✓ Alpha levels: {n_alpha_levels}")
print(f"  ✓ Steps per alpha: {steps_per_alpha}")
print(f"  ✓ Learning rate: {lr}")
print(f"  ✓ Experimentally feasible: YES")
print(f"  ✓ Requires differentiable simulator: NO")

print("\\nTraditional Optimization:")
print(f"  ✓ Total steps: {traditional_steps}")
print(f"  ✓ Learning rate: {traditional_lr}")
print(f"  ✓ Experimentally feasible: NO")
print(f"  ✓ Requires differentiable simulator: YES")

print()
print("🏆 OVERALL ASSESSMENT:")
if fm_param_error < trad_param_error and fm_traj_error < trad_traj_error:
    print("🥇 Force Matching wins on both parameter and trajectory accuracy!")
elif fm_param_error < trad_param_error:
    print("🥈 Force Matching wins on parameter accuracy, Traditional wins on trajectory accuracy")
elif fm_traj_error < trad_traj_error:
    print("🥈 Force Matching wins on trajectory accuracy, Traditional wins on parameter accuracy")
else:
    print("🥇 Traditional optimization wins on both metrics!")

print("\\n💡 KEY INSIGHTS:")
print("• Force matching can be applied to real experiments (no simulator needed)")
print("• Traditional optimization requires differentiable forward model")
print("• Gentle force matching provides smooth convergence through alpha scheduling")
print("• Both methods can achieve good parameter recovery with proper tuning")
print("• Choice depends on experimental constraints and available computational tools")


# Traditional optimization with velocity and force/accelaration as target

In [ ]:
# TRADITIONAL OPTIMIZATION WITH VELOCITY AND ACCELERATION TARGETS
print("=== TRADITIONAL OPTIMIZATION: Velocity + Acceleration Matching ===")

def velocity_acceleration_loss(p, target_traj):
    """
    Traditional optimization but matching velocities and accelerations instead of positions.
    Loss = ||v_target - v||^2 + ||a_target - a||^2
    
    This tests whether the issue with gentle force matching is the indirect objective
    or something else about the alpha scheduling approach.
    
    Args:
        p: current parameters
        target_traj: target trajectory for final mass (positions)
    
    Returns:
        loss: ||v_target - v||^2 + ||a_target - a||^2
    """
    # Forward simulation with current parameters
    states = solve_dynamics_free(p, init_state, ts)
    u_current = states[:, N-1]  # Extract final mass trajectory (positions)
    v_current = states[:, N+N-1]  # Extract final mass velocities
    
    # Compute target velocities and accelerations from target trajectory
    v_target = jnp.gradient(target_traj, ts)  # Target velocities
    a_target = jnp.gradient(v_target, ts)     # Target accelerations
    
    # Compute current accelerations
    a_current = jnp.gradient(v_current, ts)
    
    # Velocity and acceleration matching loss
    velocity_error = v_current - v_target
    acceleration_error = a_current - a_target
    
    velocity_loss = 0.5 * jnp.sum(velocity_error**2)
    acceleration_loss = 0.5 * jnp.sum(acceleration_error**2)
    
    return velocity_loss + acceleration_loss

# Create JIT-compiled versions for efficiency
vel_acc_loss_fn = jax.jit(velocity_acceleration_loss)
vel_acc_loss_grad = jax.jit(jax.grad(velocity_acceleration_loss))

print("Traditional velocity + acceleration matching implemented!")
print("- vel_acc_loss_fn: JIT-compiled loss function")
print("- vel_acc_loss_grad: JIT-compiled gradient function")
print()
print("🎯 Method:")
print("✓ Velocity matching: minimizes ||v_target - v||^2")
print("✓ Acceleration matching: minimizes ||a_target - a||^2")
print("✓ Same targets as gentle force matching but direct optimization")
print("✓ Tests if alpha scheduling is the issue vs. indirect objective")
print()


In [ ]:
# VELOCITY + ACCELERATION OPTIMIZATION LOOP
print("=== VELOCITY + ACCELERATION OPTIMIZATION LOOP ===")

# Optimization parameters
vel_acc_steps = 1000  # Total optimization steps
vel_acc_lr = 1e-2    # Learning rate

print(f"Velocity + Acceleration Matching Schedule:")
print(f"  Total steps: {vel_acc_steps}")
print(f"  Learning rate: {vel_acc_lr}")
print(f"  Method: Direct velocity and acceleration matching")
print()

# Track optimization progress
vel_acc_losses = []
vel_acc_param_errors = []
vel_acc_trajectory_errors = []
vel_acc_velocity_errors = []
vel_acc_acceleration_errors = []

# Start with same initial parameters as other methods
p_vel_acc = p_initial.copy()

print("Starting velocity + acceleration optimization...")

# Compute target velocities and accelerations once
target_v = jnp.gradient(target_traj, ts)
target_a = jnp.gradient(target_v, ts)

for step in range(vel_acc_steps):
    # Compute gradient and update parameters
    grad = vel_acc_loss_grad(p_vel_acc, target_traj)
    p_vel_acc = p_vel_acc - vel_acc_lr * grad
    
    # Compute loss
    loss = vel_acc_loss_fn(p_vel_acc, target_traj)
    vel_acc_losses.append(loss)
    
    # Track parameter error
    param_error = jnp.linalg.norm(p_vel_acc - p_true) / jnp.linalg.norm(p_true)
    vel_acc_param_errors.append(param_error)
    
    # Track trajectory error (for comparison)
    states_current = odeint(dynamics, init_state, ts, p_vel_acc, rtol=1e-12, atol=1e-12)
    x_current_last = states_current[:, N-1]
    traj_error = jnp.linalg.norm(x_current_last - target_traj) / jnp.linalg.norm(target_traj)
    vel_acc_trajectory_errors.append(traj_error)
    
    # Track velocity and acceleration errors separately
    v_current_last = states_current[:, N+N-1]
    a_current_last = jnp.gradient(v_current_last, ts)
    
    vel_error = jnp.linalg.norm(v_current_last - target_v) / jnp.linalg.norm(target_v)
    acc_error = jnp.linalg.norm(a_current_last - target_a) / jnp.linalg.norm(target_a)
    
    vel_acc_velocity_errors.append(vel_error)
    vel_acc_acceleration_errors.append(acc_error)
    
    # Print progress
    if step % 50 == 0 or step < 10 or step >= vel_acc_steps - 10:
        print(f"  Step {step:3d}/{vel_acc_steps}: loss={loss:.6f}, param_err={param_error:.4f}, traj_err={traj_error:.4f}, vel_err={vel_error:.4f}, acc_err={acc_error:.4f}")

print(f"\\n=== FINAL RESULTS (VELOCITY + ACCELERATION OPTIMIZATION) ===")
print(f"  Final loss: {vel_acc_losses[-1]:.6f}")
print(f"  Final parameter error: {vel_acc_param_errors[-1]:.4f}")
print(f"  Final trajectory error: {vel_acc_trajectory_errors[-1]:.4f}")
print(f"  Final velocity error: {vel_acc_velocity_errors[-1]:.4f}")
print(f"  Final acceleration error: {vel_acc_acceleration_errors[-1]:.4f}")
print(f"  True parameters: {p_true}")
print(f"  Final parameters: {p_vel_acc}")
print(f"\\n🎉 Velocity + Acceleration optimization completed successfully!")
print(f"Completed {vel_acc_steps} optimization steps with direct gradient descent.")
print()
print(f"🔧 Key Features:")
print(f"✓ Direct velocity matching: minimizes ||v_target - v||^2")
print(f"✓ Direct acceleration matching: minimizes ||a_target - a||^2")
print(f"✓ Same objective as gentle force matching but without alpha scheduling")
print(f"✓ Tests whether alpha scheduling is the bottleneck")
print(f"✓ Still requires differentiable simulator")


In [ ]:
# COMPREHENSIVE THREE-WAY ANALYSIS
print("\\n=== COMPREHENSIVE THREE-WAY ANALYSIS ===")

# Final metrics comparison
print("📊 FINAL PERFORMANCE COMPARISON:")
print(f"{'Metric':<25} {'Force Matching':<15} {'Position Target':<15} {'Vel+Acc Target':<15} {'Best Method':<15}")
print("-" * 95)

# Parameter error comparison
fm_param_error = param_errors_clean[-1]
trad_param_error = traditional_param_errors[-1]
vel_acc_param_error = vel_acc_param_errors[-1]
param_errors = [fm_param_error, trad_param_error, vel_acc_param_error]
param_methods = ['Force Match', 'Position', 'Vel+Acc']
param_winner = param_methods[jnp.argmin(jnp.array(param_errors))]
print(f"{'Parameter Error':<25} {fm_param_error:<15.6f} {trad_param_error:<15.6f} {vel_acc_param_error:<15.6f} {param_winner:<15}")

# Trajectory error comparison
fm_traj_error = trajectory_errors_clean[-1]
trad_traj_error = traditional_trajectory_errors[-1]
vel_acc_traj_error = vel_acc_trajectory_errors[-1]
traj_errors = [fm_traj_error, trad_traj_error, vel_acc_traj_error]
traj_winner = param_methods[jnp.argmin(jnp.array(traj_errors))]
print(f"{'Trajectory Error':<25} {fm_traj_error:<15.6f} {trad_traj_error:<15.6f} {vel_acc_traj_error:<15.6f} {traj_winner:<15}")

# Final loss comparison (note: different loss functions)
fm_final_loss = gentle_losses_clean[-1]
trad_final_loss = traditional_losses[-1]
vel_acc_final_loss = vel_acc_losses[-1]
print(f"{'Final Loss':<25} {fm_final_loss:<15.6f} {trad_final_loss:<15.6f} {vel_acc_final_loss:<15.6f} {'Different Obj':<15}")

print()
print("🔍 CONVERGENCE ANALYSIS:")

# Convergence speed (steps to reach 90% of final improvement)
fm_target = 0.9 * (fm_param_error / param_errors_clean[0])
trad_target = 0.9 * (trad_param_error / traditional_param_errors[0])
vel_acc_target = 0.9 * (vel_acc_param_error / vel_acc_param_errors[0])

convergence_steps = []
method_names = ['Force Matching', 'Position Target', 'Vel+Acc Target']
error_arrays = [param_errors_clean, traditional_param_errors, vel_acc_param_errors]
targets = [fm_target, trad_target, vel_acc_target]

for i, (errors, target) in enumerate(zip(error_arrays, targets)):
    conv_step = None
    for j, err in enumerate(errors):
        if err / errors[0] <= target:
            conv_step = j
            break
    convergence_steps.append(conv_step)
    
    if conv_step is not None:
        print(f"{method_names[i]}: {conv_step} steps to 90% convergence")
    else:
        print(f"{method_names[i]}: Did not reach 90% convergence")

# Find fastest convergence
valid_convergence = [step for step in convergence_steps if step is not None]
if valid_convergence:
    fastest_idx = convergence_steps.index(min(valid_convergence))
    print(f"Fastest convergence: {method_names[fastest_idx]}")

print()
print("🎯 METHOD CHARACTERISTICS:")

print("Force Matching (Gentle):")
print(f"  ✓ Total steps: {len(gentle_losses_clean)}")
print(f"  ✓ Alpha levels: {n_alpha_levels}")
print(f"  ✓ Steps per alpha: {steps_per_alpha}")
print(f"  ✓ Learning rate: {lr}")
print(f"  ✓ Experimentally feasible: YES")
print(f"  ✓ Requires differentiable simulator: NO")
print(f"  ✓ Local learning rule: YES")

print("\\nPosition Target (Traditional):")
print(f"  ✓ Total steps: {traditional_steps}")
print(f"  ✓ Learning rate: {traditional_lr}")
print(f"  ✓ Experimentally feasible: NO")
print(f"  ✓ Requires differentiable simulator: YES")
print(f"  ✓ Local learning rule: NO")

print("\\nVelocity + Acceleration Target:")
print(f"  ✓ Total steps: {vel_acc_steps}")
print(f"  ✓ Learning rate: {vel_acc_lr}")
print(f"  ✓ Experimentally feasible: NO")
print(f"  ✓ Requires differentiable simulator: YES")
print(f"  ✓ Local learning rule: NO")
print(f"  ✓ Same objective as Force Matching: YES")

print()
print("🏆 KEY INSIGHTS:")

# Compare velocity+acceleration vs force matching performance
if vel_acc_param_error < fm_param_error:
    improvement_ratio = fm_param_error / vel_acc_param_error
    print(f"🔍 CRITICAL FINDING: Vel+Acc target achieves {improvement_ratio:.2f}x better parameter recovery!")
    print("   → This suggests the alpha scheduling in gentle force matching is the bottleneck")
    print("   → The indirect objective (force matching) is not the main issue")
else:
    print("🔍 Force matching performs similarly to direct vel+acc optimization")
    print("   → Alpha scheduling is not the main bottleneck")
    print("   → The indirect objective might be the issue")

print()
print("💡 RECOMMENDATIONS FOR IMPROVING GENTLE FORCE MATCHING:")

if vel_acc_param_error < fm_param_error:
    print("Since direct vel+acc optimization works better:")
    print("• Improve alpha scheduling: more aggressive progression")
    print("• Reduce number of alpha levels but more steps per level")
    print("• Use adaptive alpha scheduling based on convergence")
    print("• Consider starting at higher alpha values")
else:
    print("Since force matching performs similarly to vel+acc:")
    print("• The gentle approach is fundamentally sound")
    print("• Focus on hyperparameter tuning (learning rates, steps)")
    print("• The experimental advantage may justify the small performance gap")

print()
print("🧪 EXPERIMENTAL FEASIBILITY RANKING:")
print("1. 🥇 Force Matching: Fully experimentally feasible")
print("2. 🥈 Position Target: Requires simulation")
print("3. 🥉 Vel+Acc Target: Requires simulation + derivatives")

print()
print("🎯 OPTIMIZATION PERFORMANCE RANKING:")
if vel_acc_param_error < trad_param_error < fm_param_error:
    print("1. 🥇 Vel+Acc Target: Best parameter recovery")
    print("2. 🥈 Position Target: Good performance")
    print("3. 🥉 Force Matching: Experimental advantage compensates")
elif trad_param_error < vel_acc_param_error < fm_param_error:
    print("1. 🥇 Position Target: Best parameter recovery")
    print("2. 🥈 Vel+Acc Target: Good performance")
    print("3. 🥉 Force Matching: Experimental advantage compensates")
else:
    print("Performance ranking varies - check specific metrics above")


In [ ]:
# THREE-WAY COMPARISON: Position vs Velocity+Acceleration vs Force Matching
print("=== THREE-WAY COMPARISON: All Optimization Methods ===")

fig, axes = plt.subplots(2, 4, figsize=(20, 12))

# Convert arrays for plotting
gentle_losses_np = jnp.array(gentle_losses_clean)
gentle_param_errors_np = jnp.array(param_errors_clean)
gentle_traj_errors_np = jnp.array(trajectory_errors_clean)
gentle_steps = jnp.arange(len(gentle_losses_clean))

traditional_losses_np = jnp.array(traditional_losses)
traditional_param_errors_np = jnp.array(traditional_param_errors)
traditional_traj_errors_np = jnp.array(traditional_trajectory_errors)
traditional_steps_np = jnp.arange(len(traditional_losses))

vel_acc_losses_np = jnp.array(vel_acc_losses)
vel_acc_param_errors_np = jnp.array(vel_acc_param_errors)
vel_acc_traj_errors_np = jnp.array(vel_acc_trajectory_errors)
vel_acc_steps_np = jnp.arange(len(vel_acc_losses))

# Plot 1: Loss Comparison
ax1 = axes[0, 0]
ax1.semilogy(gentle_steps, gentle_losses_np, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax1.semilogy(traditional_steps_np, traditional_losses_np, 'r-', linewidth=2, label='Position Target', alpha=0.8)
ax1.semilogy(vel_acc_steps_np, vel_acc_losses_np, 'g-', linewidth=2, label='Vel+Acc Target', alpha=0.8)
ax1.set_xlabel('Optimization Step')
ax1.set_ylabel('Loss (log scale)')
ax1.set_title('Loss Evolution Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Add vertical lines for alpha transitions in force matching
for transition in alpha_transitions_clean:
    ax1.axvline(x=transition, color='blue', linestyle=':', alpha=0.3, linewidth=1)

# Plot 2: Parameter Error Comparison
ax2 = axes[0, 1]
ax2.semilogy(gentle_steps, gentle_param_errors_np, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax2.semilogy(traditional_steps_np, traditional_param_errors_np, 'r-', linewidth=2, label='Position Target', alpha=0.8)
ax2.semilogy(vel_acc_steps_np, vel_acc_param_errors_np, 'g-', linewidth=2, label='Vel+Acc Target', alpha=0.8)
ax2.set_xlabel('Optimization Step')
ax2.set_ylabel('Parameter Error (log scale)')
ax2.set_title('Parameter Convergence Comparison')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Trajectory Error Comparison
ax3 = axes[0, 2]
ax3.semilogy(gentle_steps, gentle_traj_errors_np, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax3.semilogy(traditional_steps_np, traditional_traj_errors_np, 'r-', linewidth=2, label='Position Target', alpha=0.8)
ax3.semilogy(vel_acc_steps_np, vel_acc_traj_errors_np, 'g-', linewidth=2, label='Vel+Acc Target', alpha=0.8)
ax3.set_xlabel('Optimization Step')
ax3.set_ylabel('Trajectory Error (log scale)')
ax3.set_title('Trajectory Matching Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Velocity + Acceleration Error Breakdown
ax4 = axes[0, 3]
vel_acc_vel_errors_np = jnp.array(vel_acc_velocity_errors)
vel_acc_acc_errors_np = jnp.array(vel_acc_acceleration_errors)
ax4.semilogy(vel_acc_steps_np, vel_acc_vel_errors_np, 'g-', linewidth=2, label='Velocity Error', alpha=0.8)
ax4.semilogy(vel_acc_steps_np, vel_acc_acc_errors_np, 'orange', linewidth=2, label='Acceleration Error', alpha=0.8)
ax4.set_xlabel('Optimization Step')
ax4.set_ylabel('Error (log scale)')
ax4.set_title('Vel+Acc Method: Component Errors')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Plot 5: Final Trajectory Comparison
ax5 = axes[1, 0]
# Get final trajectories
states_gentle_final = odeint(dynamics, init_state, ts, p_current, rtol=1e-12, atol=1e-12)
x_gentle_final = states_gentle_final[:, N-1]

states_traditional_final = odeint(dynamics, init_state, ts, p_traditional, rtol=1e-12, atol=1e-12)
x_traditional_final = states_traditional_final[:, N-1]

states_vel_acc_final = odeint(dynamics, init_state, ts, p_vel_acc, rtol=1e-12, atol=1e-12)
x_vel_acc_final = states_vel_acc_final[:, N-1]

ax5.plot(ts, target_traj, 'k-', linewidth=3, label='Target', alpha=0.9)
ax5.plot(ts, x_gentle_final, 'b--', linewidth=2, label='Force Matching', alpha=0.8)
ax5.plot(ts, x_traditional_final, 'r:', linewidth=2, label='Position Target', alpha=0.8)
ax5.plot(ts, x_vel_acc_final, 'g-.', linewidth=2, label='Vel+Acc Target', alpha=0.8)
ax5.set_xlabel('Time')
ax5.set_ylabel('Position')
ax5.set_title('Final Trajectory Comparison')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Plot 6: Parameter Recovery Comparison
ax6 = axes[1, 1]
param_names = [f'p[{i}]' for i in range(len(p_true))]
x_pos = jnp.arange(len(p_true))
width = 0.15

ax6.bar(x_pos - width, p_true, width, label='True Parameters', alpha=0.8, color='black')
ax6.bar(x_pos, p_current, width, label='Force Matching', alpha=0.8, color='blue')
ax6.bar(x_pos + width, p_traditional, width, label='Position Target', alpha=0.8, color='red')
ax6.bar(x_pos + 2*width, p_vel_acc, width, label='Vel+Acc Target', alpha=0.8, color='green')

ax6.set_xlabel('Parameter Index')
ax6.set_ylabel('Parameter Value')
ax6.set_title('Parameter Recovery Comparison')
ax6.set_xticks(x_pos + width/2)
ax6.set_xticklabels(param_names)
ax6.legend()
ax6.grid(True, alpha=0.3)

# Plot 7: Convergence Rate Analysis
ax7 = axes[1, 2]
gentle_improvement = gentle_param_errors_np[0] / gentle_param_errors_np
traditional_improvement = traditional_param_errors_np[0] / traditional_param_errors_np
vel_acc_improvement = vel_acc_param_errors_np[0] / vel_acc_param_errors_np

ax7.plot(gentle_steps, gentle_improvement, 'b-', linewidth=2, label='Force Matching', alpha=0.8)
ax7.plot(traditional_steps_np, traditional_improvement, 'r-', linewidth=2, label='Position Target', alpha=0.8)
ax7.plot(vel_acc_steps_np, vel_acc_improvement, 'g-', linewidth=2, label='Vel+Acc Target', alpha=0.8)
ax7.set_xlabel('Optimization Step')
ax7.set_ylabel('Parameter Error Improvement (ratio)')
ax7.set_title('Convergence Rate Comparison')
ax7.legend()
ax7.grid(True, alpha=0.3)

# Plot 8: Final Error Comparison (Bar Chart)
ax8 = axes[1, 3]
methods = ['Force\\nMatching', 'Position\\nTarget', 'Vel+Acc\\nTarget']
final_param_errors = [gentle_param_errors_np[-1], traditional_param_errors_np[-1], vel_acc_param_errors_np[-1]]
final_traj_errors = [gentle_traj_errors_np[-1], traditional_traj_errors_np[-1], vel_acc_traj_errors_np[-1]]

x_methods = jnp.arange(len(methods))
width = 0.35

bars1 = ax8.bar(x_methods - width/2, final_param_errors, width, label='Parameter Error', alpha=0.8, color='lightblue')
bars2 = ax8.bar(x_methods + width/2, final_traj_errors, width, label='Trajectory Error', alpha=0.8, color='lightcoral')

ax8.set_xlabel('Method')
ax8.set_ylabel('Final Error')
ax8.set_title('Final Error Comparison')
ax8.set_xticks(x_methods)
ax8.set_xticklabels(methods)
ax8.legend()
ax8.grid(True, alpha=0.3)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax8.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax8.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()

# Save the comparison plot
if save_flag:
    plt.savefig(f"{output_dir}/three_way_method_comparison.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_dir}/three_way_method_comparison.pdf", bbox_inches='tight')
    print(f"Saved plot: three_way_method_comparison.png and .pdf")

plt.show()
